# K-Means Clustering — From Scratch (Pure Python + NumPy)

## What is K-Means?
K-Means partitions N data points into **K clusters** so that each point belongs to the cluster with the nearest centroid.

### The Algorithm (Lloyd's Algorithm)
```
1. Initialise: randomly pick K points as centroids
2. Assign:     each point → nearest centroid (by Euclidean distance)
3. Move:       each centroid → mean of all points assigned to it
4. Repeat steps 2-3 until centroids stop moving (convergence)
```

### Key Properties
| Property | Value |
|----------|-------|
| Type | Unsupervised — no labels needed |
| Output | Cluster assignment for every point |
| Convergence | Guaranteed (monotonically decreasing WCSS) |
| Complexity | O(n × K × iterations) |
| Weakness | Sensitive to initialisation; must choose K in advance |

### This notebook implements KMeans from scratch using only:
- `numpy` — for distance maths and mean computation
- `random` — for random centroid initialisation


## Imports

In [ ]:
import random
import numpy as np

## The KMeans Class

### `__init__` — Constructor
Stores two hyperparameters:
- `n_clusters` (K) — number of clusters to form
- `max_iter` — maximum number of assign-move iterations (safety cap to avoid infinite loops)
- `centroids` — will hold the K centroid coordinates after fitting


In [ ]:
class KMeans:
    def __init__(self, n_clusters=2, max_iter=100):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.centroids = None

### `fit_predict` — Main Method
Runs the full K-Means loop and returns cluster labels for every data point.

**Step-by-step:**
1. **Initialise centroids** — randomly sample K row indices from X; use those rows as starting centroids
2. **Loop** (up to `max_iter` times):
   - `assign_clusters(X)` → assign each point to nearest centroid
   - `move_centroids(X, cluster_group)` → recompute centroid positions
   - **Convergence check** — if centroids didn't move at all, stop early
3. Return the final cluster assignments


In [ ]:
    def fit_predict(self, X):
        # Step 1: Random initialisation — pick K rows as starting centroids
        random_index = random.sample(range(0, X.shape[0]), self.n_clusters)
        self.centroids = X[random_index]

        for i in range(self.max_iter):
            # Step 2a: Assign each point to the nearest centroid
            cluster_group = self.assign_clusters(X)
            old_centroids = self.centroids

            # Step 2b: Move centroids to the mean of their assigned points
            self.centroids = self.move_centroids(X, cluster_group)

            # Step 2c: Convergence check — stop if nothing changed
            if (old_centroids == self.centroids).all():
                break

        return cluster_group

### `assign_clusters` — Nearest Centroid Assignment

For every data point, compute its **Euclidean distance** to each centroid and assign it to the closest one.

**Euclidean distance formula:**
```
d(a, b) = sqrt( Σ (aᵢ - bᵢ)² )
         = sqrt( dot(a-b, a-b) )
```
Using `np.dot(row - centroid, row - centroid)` is equivalent to `np.sum((row - centroid)**2)` — just written as a dot product.

The index of the minimum distance is the cluster label for that point.


In [ ]:
    def assign_clusters(self, X):
        cluster_group = []
        distances = []

        for row in X:
            for centroid in self.centroids:
                # Euclidean distance via dot product: sqrt(||row - centroid||²)
                distances.append(np.sqrt(np.dot(row - centroid, row - centroid)))
            min_distance = min(distances)
            index_pos = distances.index(min_distance)
            cluster_group.append(index_pos)
            distances.clear()

        return np.array(cluster_group)

### `move_centroids` — Recompute Centroid Positions

For each cluster, compute the **mean** of all points assigned to it.
This new mean becomes the updated centroid position.

**Why the mean?** The mean minimises the sum of squared distances from all points in the cluster to the centroid (this is exactly WCSS — Within-Cluster Sum of Squares).


In [ ]:
    def move_centroids(self, X, cluster_group):
        new_centroids = []
        cluster_type = np.unique(cluster_group)

        for type in cluster_type:
            # Mean of all points belonging to this cluster
            new_centroids.append(X[cluster_group == type].mean(axis=0))

        return np.array(new_centroids)

## Quick Demo — Test the KMeans Class

In [ ]:
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt

# Generate 2D synthetic data with 3 clusters
X_demo, _ = make_blobs(n_samples=150, centers=3, cluster_std=0.8, random_state=42)

km_demo = KMeans(n_clusters=3, max_iter=300)
labels = km_demo.fit_predict(X_demo)

plt.figure(figsize=(7, 5))
for k in range(3):
    plt.scatter(X_demo[labels == k, 0], X_demo[labels == k, 1], label=f'Cluster {k}')
plt.scatter(km_demo.centroids[:, 0], km_demo.centroids[:, 1],
            marker='X', s=200, c='black', label='Centroids')
plt.title('KMeans From Scratch — 3 Clusters')
plt.legend()
plt.tight_layout()
plt.show()
print(f"Converged. Centroids:\n{km_demo.centroids}")

## Summary

```
KMeans.fit_predict(X):

  1. centroids = X[random K rows]           ← random init

  repeat max_iter times:
    2. for each point x:
         distances = [euclidean(x, c) for c in centroids]
         label[x] = argmin(distances)        ← assign step

    3. for each cluster k:
         centroids[k] = mean(X[label == k])  ← move step

    4. if centroids unchanged → break        ← convergence

  return labels

Weakness of random init → solved by KMeans++ (sklearn default)
  → smarter initialisation spreads centroids far apart
```
